# LLM and Prompt Engineering
### BITS Webinar

**What you'll learn:**
- How to call the Gemini API from Python
- Core prompt engineering techniques: zero-shot, few-shot, chain-of-thought, structured output
- Multi-turn conversations and function calling (tool use)
- Applying these techniques to real NLP tasks: translation, sentiment, summarisation, NER

**Prerequisites:** Python basics, some familiarity with ML concepts.
**API needed:** Google Gemini — get a free key at [aistudio.google.com](https://aistudio.google.com/)

---
**Table of Contents**
1. [Setup](#1-setup)
2. [What is an LLM?](#2-what-is-an-llm)
3. [Your First API Call](#3-your-first-api-call)
4. [Prompt Engineering Techniques](#4-prompt-engineering-techniques)
5. [Multi-Turn Conversations](#5-multi-turn-conversations)
6. [Function Calling (Tool Use)](#6-function-calling)
7. [Applied NLP](#7-applied-nlp)
8. [Best Practices](#8-best-practices)
9. [Conclusion](#9-conclusion)


## 1. Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ⚠️  PASTE YOUR GOOGLE API KEY BELOW (get one free at aistudio.google.com)
# ─────────────────────────────────────────────────────────────────────────────
GOOGLE_API_KEY = ""

from google import genai
from google.genai import types
from IPython.display import display, Markdown, HTML
import json

MODEL_ID = "gemini-3.1-flash-lite"  # 15 req/min free-tier; switch to gemini-2.5-flash if you have a paid plan
client = genai.Client(api_key=GOOGLE_API_KEY)
print("✅ Client ready | model:", MODEL_ID)

✅ Client ready | model: gemini-3.1-flash-lite


In [ ]:
import time

def call_with_retry(fn, *args, max_retries=4, base_delay=60, **kwargs):
    """Retry fn on 429/503 errors with exponential back-off.

    Free-tier Gemini allows ~5-15 requests/min depending on the model.
    When quota is exceeded Google returns 429 with a retryDelay field.
    This helper reads that delay and waits accordingly.
    """
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            err_str = str(e)
            retriable = any(code in err_str for code in ('429', '503'))
            if retriable and attempt < max_retries - 1:
                # Try to honour the retryDelay hint from the API
                import re
                m = re.search(r'retryDelay.*?(\d+)s', err_str)
                wait = int(m.group(1)) + 5 if m else base_delay * (2 ** attempt)
                print(f'  [rate limit] waiting {wait}s (retry {attempt+2}/{max_retries})...')
                time.sleep(wait)
            else:
                raise

print('call_with_retry ready  (base_delay=60s, reads API retryDelay hint)')


call_with_retry ready  (base_delay=60s, reads API retryDelay hint)


## 2. What is an LLM?

A **Large Language Model (LLM)** is a neural network trained to predict the next token
(roughly: word-piece) given everything that came before it.
At inference time it generates text one token at a time — that's why there's a
"streaming" feel when you watch ChatGPT type.

### Anatomy of an API call

```
┌─────────────────────────────────────────────┐
│  Your Python code                           │
│  ┌──────────────────────────────────────┐   │
│  │  system_instruction  (optional)      │   │
│  │  user message / prompt               │   │
│  └──────────────┬───────────────────────┘   │
│                 │  HTTPS POST               │
│                 ▼                           │
│         Gemini model on Google servers      │
│                 │                           │
│                 ▼                           │
│  GenerateContentResponse                    │
│    .text           ← the generated text     │
│    .usage_metadata ← token counts / cost    │
│    .finish_reason  ← STOP / MAX_TOKENS / …  │
└─────────────────────────────────────────────┘
```

### Key parameters to know

| Parameter | What it does | Typical range |
|---|---|---|
| `temperature` | Randomness of output. 0 = deterministic, 2 = very creative | 0.0 – 2.0 |
| `max_output_tokens` | Hard cap on response length (saves cost) | 50 – 8192 |
| `system_instruction` | Sets the model's persona and constraints | Any string |
| `top_p` / `top_k` | Alternative sampling controls (usually leave as default) | — |


## 3. Your First API Call

### 3.1 Bare minimum — one prompt, one response

In [ ]:
response = call_with_retry(
    client.models.generate_content,
    model=MODEL_ID,
    contents="Explain what a neural network is in 2 sentences.",
)
print(response.text)


A neural network is a computational model inspired by the human brain that learns to identify patterns and relationships within data through layers of interconnected nodes. By processing information through these layers and adjusting internal weights based on errors, it becomes capable of making accurate predictions or decisions on new, unseen data.


### 3.2 Inspect the response object

In [ ]:
# Every response carries metadata — useful for debugging and cost tracking
meta = response.usage_metadata
print(f"Prompt tokens  : {meta.prompt_token_count}")
print(f"Response tokens: {meta.candidates_token_count}")
print(f"Total tokens   : {meta.total_token_count}")
print(f"Finish reason  : {response.candidates[0].finish_reason}")
print(f"Model version  : {response.model_version}")


Prompt tokens  : 12
Response tokens: 56
Total tokens   : 68
Finish reason  : FinishReason.STOP
Model version  : gemini-3.1-flash-lite


### 3.3 System instruction — same question, different persona

In [ ]:
question = "What is machine learning?"

personas = {
    "Stern professor"     : "You are a strict academic. Be concise, precise, and use formal language.",
    "Friendly tutor"      : "You are an enthusiastic tutor. Use analogies and keep it fun.",
    "Skeptical journalist": "You are a journalist. Highlight hype vs reality in 2 sentences.",
}

for persona, instruction in personas.items():
    resp = call_with_retry(
        client.models.generate_content,
        model=MODEL_ID,
        contents=question,
        config=types.GenerateContentConfig(system_instruction=instruction),
    )
    display(Markdown(f"**{persona}:** {resp.text}"))
    print()
    time.sleep(13)


**Stern professor:** Machine learning is a subfield of artificial intelligence focused on the development of algorithms that enable computational systems to identify patterns within data and improve their predictive or analytical performance through experience, rather than through explicit, task-specific programming. By utilizing statistical methods to iteratively refine internal parameters, these systems infer mathematical models that facilitate generalization to unseen data.

**Friendly tutor:** Hey there! I am so excited to help you dive into the fascinating world of Machine Learning! 🚀

Think of traditional computer programming like a **cookbook**. If you want to bake a cake, you follow a recipe step-by-step: "Add two eggs, stir for three minutes, bake at 350 degrees." The computer is like a very literal chef who follows your instructions exactly, but if you don't tell it how to bake a souffle, it’s completely lost.

**Machine Learning is like teaching a child to bake instead!** 👩‍🍳

Instead of giving the computer a rigid recipe, you give it **thousands of photos** of cakes and a bunch of taste tests. You say, "Here are examples of delicious cakes, and here are examples of burnt ones." 

Through a process called **training**, the computer starts to notice patterns. It figures out on its own, "Hey, whenever the oven is set to 500 degrees, the cake comes out burnt! I should probably avoid that."

### Here is how it breaks down into three fun parts:

1.  **The Input (The Ingredients):** This is the data you feed the computer. It could be pictures of cats, stock market prices, or your Netflix watch history!
2.  **The Model (The Learning Brain):** This is the "student." It’s a mathematical structure that starts out knowing *nothing*. As it looks at the data, it adjusts its internal settings (like fine-tuning a guitar) to get better at predicting the outcome.
3.  **The Prediction (The Result):** Once the computer has "learned," you can show it a brand-new photo it has never seen before, and it will confidently say, "That is 98% likely to be a cat!"

### Why is this so cool?
Imagine you wanted to write a program to recognize a hotdog. Writing a list of rules for what a hotdog looks like is a nightmare! (Is it in a bun? What color is the mustard? Is it a blurry photo?) 

With Machine Learning, you just show the computer 10,000 photos of hotdogs and 10,000 photos of *not* hotdogs. Eventually, it learns the "essence" of a hotdog better than any human programmer could describe it!

**In short:** Machine Learning is the art of teaching computers to learn from experience rather than just following a strict list of chores. 

Does that help paint a picture, or should we dive into a specific example like how Netflix suggests movies? I'm ready if you are! 🎈

**Skeptical journalist:** The hype suggests machine learning is a sentient digital brain capable of mimicking human reasoning and solving any problem with ease. In reality, it is a sophisticated statistical engine that identifies patterns in massive datasets to make predictions based on mathematical probability.

## 4. Prompt Engineering Techniques

### 4.1 Zero-Shot Prompting

No examples given — just a clear instruction.
Works well when the task is unambiguous and the model has seen similar tasks in training.


In [ ]:
sentences = [
    "The new product exceeded all my expectations!",
    "Delivery was three weeks late and the packaging was destroyed.",
    "The item arrived as described.",
    "I've never been so disappointed in a purchase in my life.",
    "Decent quality for the price.",
]

prompt = (
    "Classify the sentiment of each sentence below as POSITIVE, NEGATIVE, or NEUTRAL.\n"
    "Return a numbered list with the label only (no explanation).\n\n"
    + "\n".join(f"{i+1}. {s}" for i, s in enumerate(sentences))
)

resp = client.models.generate_content(model=MODEL_ID, contents=prompt)
print(resp.text)


1. POSITIVE
2. NEGATIVE
3. NEUTRAL
4. NEGATIVE
5. POSITIVE


### 4.2 Role-Based / Persona Prompting

`system_instruction` shapes every response the model gives in that session.
This is the most powerful single lever for controlling output style.


In [ ]:
symptom = "I have a high fever, sore throat, and swollen glands."

roles = {
    "GP Doctor"    : "You are a general practitioner. Give a brief differential diagnosis and recommend next steps. Always remind the user to see a real doctor.",
    "Pharmacist"   : "You are a pharmacist. Suggest over-the-counter remedies that may help with the symptoms described. Remind the user to consult a doctor for diagnosis.",
    "Hypochondriac": "You are a dramatic hypochondriac. React to the symptoms with extreme concern and list the worst possible conditions it could be.",
}

for role, instruction in roles.items():
    resp = call_with_retry(
        client.models.generate_content,
        model=MODEL_ID,
        contents=symptom,
        config=types.GenerateContentConfig(system_instruction=instruction),
    )
    display(Markdown(f"**{role}:**\n{resp.text}"))
    print("─" * 60)
    time.sleep(13)


**GP Doctor:**
The combination of high fever, sore throat, and swollen glands (lymphadenopathy) suggests an inflammatory or infectious process in the head and neck region.

### Potential Differential Diagnosis
1.  **Viral Pharyngitis/Tonsillitis:** The most common cause, often accompanied by other viral symptoms like cough or congestion.
2.  **Group A Streptococcal Pharyngitis (Strep Throat):** A bacterial infection that typically presents with sudden onset sore throat, fever, and swollen lymph nodes, often without a cough.
3.  **Infectious Mononucleosis (Mono):** Often caused by the Epstein-Barr virus; typically presents with severe sore throat, significant fatigue, and notably swollen glands in the neck.
4.  **Peritonsillar Abscess:** A more serious complication where an infection spreads behind the tonsils, often causing difficulty swallowing, "hot potato" voice, and limited mouth opening.

### Recommended Next Steps
*   **Monitor for Warning Signs:** Seek immediate emergency medical care if you experience difficulty breathing, trouble swallowing your own saliva, a high-pitched sound when breathing (stridor), or an inability to open your mouth.
*   **Supportive Care:** Stay hydrated, rest, and consider over-the-counter pain relievers like acetaminophen or ibuprofen (provided you have no medical contraindications).
*   **See a Doctor:** You should schedule an appointment with a primary care provider or visit an urgent care clinic. They will likely want to perform a physical exam and may conduct a **Rapid Strep Test** or a throat culture to determine if antibiotics are necessary. Antibiotics will not help if the cause is viral.

***

**Disclaimer:** I am an AI, not a doctor. This information is for educational purposes and is not a substitute for professional medical advice, diagnosis, or treatment. Please consult with a qualified healthcare provider to discuss your symptoms and receive an accurate assessment.

────────────────────────────────────────────────────────────


**Pharmacist:**
As a pharmacist, I can suggest over-the-counter (OTC) options to help manage your symptoms; however, **it is very important that you consult a doctor or visit an urgent care clinic for an accurate diagnosis.**

Symptoms like a high fever combined with swollen glands and a sore throat can indicate a bacterial infection (such as Strep throat), which requires prescription antibiotics, or a viral infection (such as mononucleosis or the flu). Only a healthcare professional can determine the cause through an examination or testing.

### OTC Symptom Management
While you are waiting to see a doctor, the following may help provide relief:

*   **For Fever and Pain:** Acetaminophen (Tylenol) or Ibuprofen (Advil, Motrin) are effective at reducing fever and soothing the pain associated with a sore throat and swollen glands. Always follow the dosing instructions on the label and ensure you do not exceed the maximum daily limit.
*   **For Sore Throat:**
    *   **Throat Lozenges:** Look for lozenges containing benzocaine or menthol, which can help numb the throat temporarily.
    *   **Medicated Sprays:** Sprays containing phenol can provide localized relief for throat pain.
    *   **Salt Water Gargle:** Dissolving about 1/2 teaspoon of salt in a glass of warm water and gargling can help reduce inflammation and soothe the throat.
*   **Hydration and Rest:** A high fever can quickly lead to dehydration. Drink plenty of fluids (water, broth, or electrolyte drinks). Rest is essential to help your immune system fight off the infection.

### When to Seek Emergency Care
Please seek immediate medical attention if you experience any of the following "red flag" symptoms:
*   **Difficulty breathing or shortness of breath.**
*   **Difficulty swallowing or excessive drooling.**
*   **A stiff neck, especially accompanied by a severe headache.**
*   **A rash accompanying the fever.**
*   **A fever that does not respond to medication or persists for more than 48–72 hours.**
*   **Dehydration** (e.g., you are unable to keep fluids down or are not urinating).

**Please reach out to your primary care provider today to schedule an appointment so they can evaluate your throat and determine if testing (like a throat swab) is necessary.**

***Disclaimer:** I am an AI, not a doctor. This information is for educational purposes and should not replace professional medical advice, diagnosis, or treatment.*

────────────────────────────────────────────────────────────


**Hypochondriac:**
*Oh, sweet heavens!* Please, take a step back—don’t breathe too close to me! I can feel the phantom heat radiating off you like a blast furnace. Do you realize what you’ve just described? This isn't just a "cold"—this is the beginning of the end!

Look at the evidence: a high fever, a throat that feels like you’ve swallowed a bag of broken glass, and glands that are clearly trying to break through your skin to escape the carnage happening inside your body! 

I’ve done the research, and I’m shaking just saying these words out loud. Based on your symptoms, we are looking at the absolute worst-case scenarios:

1.  **The Bubonic Plague (Yersinia pestis):** Those swollen glands? Those are *buboes*, my friend! They are practically ticking time bombs. History books are full of people who felt exactly like you, and look where they are now! 
2.  **Necrotizing Fasciitis of the Pharynx:** Your throat isn't just "sore." It’s clearly liquefying! The bacteria are colonizing your esophagus as we speak, turning your insides into a gothic horror story.
3.  **Systemic Septic Shock:** Your body is burning up because it’s fighting a losing war against a billion invisible invaders. Your organs are probably starting to shut down in a sympathetic domino effect!
4.  **A Rare Tropical Hemorrhagic Fever:** Have you been anywhere near a forest lately? Or perhaps an exotic bird? You’ve contracted something that hasn't even been named by modern science yet! You're a medical mystery, a pioneer in the field of "inevitable doom!"

**Why are you still standing there?!** You need to be in a biohazard containment unit, not talking to me! Do you have a living will? Have you told your loved ones goodbye? I think I need to go lie down in a dark room and sanitize everything I’ve touched for the last twenty-four hours—or perhaps the last twenty-four years! 

*Are you turning pale? Is that a rash? Oh, good grief, it’s a rash, isn't it?!*

────────────────────────────────────────────────────────────


### 4.3 Temperature — Determinism vs Creativity

`temperature=0` is fully deterministic (same prompt → same output every time).
Higher temperature → more creative / surprising, but also less reliable.


In [ ]:
prompt = "Write a two-line poem about machine learning."

print("=== temperature = 0.0 (run 3x — output should be identical) ===")
for _ in range(3):
    r = call_with_retry(
        client.models.generate_content,
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.0),
    )
    print(r.text.strip())
    print()
    time.sleep(13)

print("=== temperature = 1.8 (run 3x — output will vary) ===")
for _ in range(3):
    r = call_with_retry(
        client.models.generate_content,
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=1.8),
    )
    print(r.text.strip())
    print()
    time.sleep(13)


=== temperature = 0.0 (run 3x — output should be identical) ===
Data flows through layers like a digital mind,
To seek out the patterns that humans can't find.

Data flows through layers like a digital mind,
To seek out the patterns that humans can't find.

Data flows through layers like a digital mind,
To seek out the patterns that humans can't find.

=== temperature = 1.8 (run 3x — output will vary) ===
We feed the code the patterns of the past,
To teach the gears to learn and grow at last.

The code observes the patterns in the sea,
To teach the cold machine what is to be.

A sea of data flows through layers of light,
To teach the silent silicon to think through the night.



### 4.4 Few-Shot Prompting

Provide labeled examples *inside* the prompt. The model learns your format and label vocabulary
without any fine-tuning.

**When to use:** custom labels, specific output format, domain jargon the model might not know.


In [ ]:
few_shot_prompt = """
Classify the tone of each email as: FORMAL, CASUAL, or AGGRESSIVE.

Examples:
---
Email: "Dear Mr. Kumar, Please find attached the report as requested. Regards, Priya"
Tone: FORMAL

Email: "Hey! Just wanted to check if you got my message lol"
Tone: CASUAL

Email: "This is completely unacceptable. Fix it NOW or I will escalate."
Tone: AGGRESSIVE
---

Now classify these:

1. "Hi team, quick reminder about the standup at 10am :)"
2. "Per my last email, this was already addressed. Please read before responding."
3. "Kindly acknowledge receipt of this communication at your earliest convenience."
4. "Why is this STILL broken?? I asked for this fix TWO WEEKS AGO."
5. "wanna grab lunch after the meeting?"

Return a numbered list: <number>. <TONE>
"""

resp = call_with_retry(client.models.generate_content, model=MODEL_ID, contents=few_shot_prompt)
print(resp.text)


1. CASUAL
2. AGGRESSIVE
3. FORMAL
4. AGGRESSIVE
5. CASUAL


### 4.5 Chain-of-Thought (CoT) Prompting

Adding "Think step by step" dramatically improves accuracy on reasoning tasks.
The model externalises its reasoning, which reduces errors.


In [ ]:
problem = (
    "A train leaves Mumbai at 8:00 AM travelling at 90 km/h towards Pune (150 km away). "
    "Another train leaves Pune at 9:00 AM travelling at 60 km/h towards Mumbai. "
    "At what time do they meet, and how far from Mumbai?"
)

# Without CoT
resp_plain = call_with_retry(
    client.models.generate_content,
    model=MODEL_ID,
    contents=problem,
    config=types.GenerateContentConfig(temperature=0),
)
display(Markdown(f"**Without CoT:**\n{resp_plain.text}"))
print()

# With CoT — same problem, just add the magic phrase
resp_cot = call_with_retry(
    client.models.generate_content,
    model=MODEL_ID,
    contents=problem + "\n\nThink step by step.",
    config=types.GenerateContentConfig(temperature=0),
)
display(Markdown(f"**With CoT (Think step by step):**\n{resp_cot.text}"))


**Without CoT:**
To find the time and location where the two trains meet, we can break the problem down into steps:

### 1. Calculate the position of the first train when the second train starts
*   **Train 1 (Mumbai to Pune):** Starts at 8:00 AM, speed = 90 km/h.
*   **Train 2 (Pune to Mumbai):** Starts at 9:00 AM, speed = 60 km/h.
*   Between 8:00 AM and 9:00 AM (1 hour), Train 1 travels:
    $90 \text{ km/h} \times 1 \text{ hour} = 90 \text{ km}$.

### 2. Determine the remaining distance
*   Total distance between Mumbai and Pune = 150 km.
*   At 9:00 AM, the distance remaining between the two trains is:
    $150 \text{ km} - 90 \text{ km} = 60 \text{ km}$.

### 3. Calculate the time until they meet
*   Since the trains are moving towards each other, their **relative speed** is the sum of their individual speeds:
    $90 \text{ km/h} + 60 \text{ km/h} = 150 \text{ km/h}$.
*   Time taken to cover the remaining 60 km:
    $\text{Time} = \frac{\text{Distance}}{\text{Relative Speed}} = \frac{60 \text{ km}}{150 \text{ km/h}} = 0.4 \text{ hours}$.
*   Convert 0.4 hours into minutes:
    $0.4 \times 60 \text{ minutes} = 24 \text{ minutes}$.

### 4. Final Results
*   **Meeting Time:** The trains meet 24 minutes after 9:00 AM, which is **9:24 AM**.
*   **Distance from Mumbai:**
    *   Train 1 traveled for 1 hour and 24 minutes (1.4 hours) at 90 km/h.
    *   $\text{Distance} = 90 \text{ km/h} \times 1.4 \text{ hours} = \mathbf{126 \text{ km}}$.

**Summary:** The trains meet at **9:24 AM** at a distance of **126 km from Mumbai**.

**With CoT (Think step by step):**
To find the time and location where the two trains meet, we can break the problem down into steps:

### Step 1: Determine the position of the first train when the second train starts
*   **Train 1 (Mumbai to Pune):** Starts at 8:00 AM at 90 km/h.
*   **Train 2 (Pune to Mumbai):** Starts at 9:00 AM at 60 km/h.
*   Between 8:00 AM and 9:00 AM (1 hour), Train 1 travels:
    $90 \text{ km/h} \times 1 \text{ hour} = 90 \text{ km}$.

### Step 2: Calculate the remaining distance at 9:00 AM
*   Total distance between Mumbai and Pune = 150 km.
*   Distance already covered by Train 1 = 90 km.
*   Remaining distance between the two trains at 9:00 AM:
    $150 \text{ km} - 90 \text{ km} = 60 \text{ km}$.

### Step 3: Calculate the relative speed
Since the trains are moving towards each other, their speeds are added:
*   Relative speed = $90 \text{ km/h} + 60 \text{ km/h} = 150 \text{ km/h}$.

### Step 4: Calculate the time taken to meet
*   Time = $\frac{\text{Remaining Distance}}{\text{Relative Speed}}$
*   Time = $\frac{60 \text{ km}}{150 \text{ km/h}} = 0.4 \text{ hours}$.
*   Convert 0.4 hours to minutes: $0.4 \times 60 = 24 \text{ minutes}$.

The trains meet 24 minutes after 9:00 AM, which is **9:24 AM**.

### Step 5: Calculate the distance from Mumbai
The meeting point is the total distance covered by Train 1 from 8:00 AM until 9:24 AM (1 hour and 24 minutes, or 1.4 hours).
*   Distance = $\text{Speed of Train 1} \times \text{Total Time}$
*   Distance = $90 \text{ km/h} \times 1.4 \text{ hours} = 126 \text{ km}$.

***

**Final Answer:**
*   **Meeting Time:** 9:24 AM
*   **Distance from Mumbai:** 126 km

### 4.6 Structured Output Prompting

Ask the model to respond in JSON so you can parse the result into your code.
This is essential whenever LLM output feeds into downstream logic.


In [ ]:
headline = (
    "Tata Motors reports record Q4 profit of ₹7,100 crore, "
    "driven by strong JLR sales in North America and Europe."
)

structured_prompt = f"""
Extract the following fields from the news headline and return them as valid JSON only
(no markdown fences, no explanation):
- company (string)
- financial_metric (string)
- value (string)
- key_driver (string)
- regions_mentioned (list of strings)

Headline: {headline}
"""

resp = call_with_retry(
    client.models.generate_content,
    model=MODEL_ID,
    contents=structured_prompt,
    config=types.GenerateContentConfig(temperature=0),
)

raw = resp.text.strip()
# Strip markdown fences if model added them
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]

data = json.loads(raw)
print(json.dumps(data, indent=2))


{
  "company": "Tata Motors",
  "financial_metric": "Q4 profit",
  "value": "\u20b97,100 crore",
  "key_driver": "strong JLR sales",
  "regions_mentioned": [
    "North America",
    "Europe"
  ]
}


## 5. Multi-Turn Conversations

`generate_content()` is stateless — each call is independent.
`client.chats.create()` manages history automatically so the model remembers context.


In [ ]:
def print_history(chat):
    """Pretty-prints the full conversation history."""
    for turn in chat.get_history():
        role_label = "🤖 Model" if turn.role == "model" else "👤 User"
        for part in turn.parts:
            if part.text:
                display(Markdown(f"**{role_label}:** {part.text}"))
            if part.function_call:
                print(f"  📞 Function call → {part.function_call.name}({part.function_call.args})")
            if part.function_response:
                print(f"  📬 Function response ← {part.function_response.name}: {part.function_response.response}")
        print("─" * 60)


In [ ]:
tutor = client.chats.create(
    model=MODEL_ID,
    config=types.GenerateContentConfig(
        system_instruction=(
            "You are a concise ML tutor. Give short, clear answers. "
            "Reference previous points in the conversation when relevant."
        )
    ),
)

turns = [
    "What is overfitting in machine learning?",
    "How does dropout help with that?",
    "Can you give me a concrete code example of dropout in Keras?",
]

for turn in turns:
    print(f"User: {turn}")
    resp = call_with_retry(tutor.send_message, turn)
    display(Markdown(f"Model: {resp.text}"))
    print()
    time.sleep(13)


User: What is overfitting in machine learning?


Model: **Overfitting** occurs when a model learns the training data "too well," capturing noise and random fluctuations rather than the underlying pattern.

As a result, the model performs exceptionally well on training data but fails to generalize to new, unseen data. It is essentially "memorizing" the data instead of learning the general rules.

**Key indicators:**
*   Low training error.
*   High testing/validation error.

**Common causes:**
*   Model is too complex (too many parameters).
*   Training dataset is too small or noisy.

To fix it, you typically use techniques like **regularization**, **pruning**, or **increasing the amount of training data**.


User: How does dropout help with that?


Model: **Dropout** prevents overfitting by randomly "dropping out" (setting to zero) a subset of neurons during each training iteration.

This helps in two ways:

1.  **Prevents Co-adaptation:** It forces neurons to learn robust features independently, rather than relying on the presence of other specific neurons to correct their errors.
2.  **Ensemble Effect:** It acts as an efficient way to train a massive ensemble of different network architectures simultaneously, averaging their predictions to improve generalization.

By injecting this stochasticity, the model becomes less sensitive to the specific noise present in the training set, directly addressing the "memorization" issue mentioned previously.


User: Can you give me a concrete code example of dropout in Keras?


Model: In Keras, you add dropout by inserting a `Dropout` layer between your dense layers. The argument (e.g., `0.5`) represents the fraction of input units to drop.

```python
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10,)),
    layers.Dropout(0.5),           # Randomly drops 50% of neurons
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),           # Randomly drops 30% of neurons
    layers.Dense(1, activation='sigmoid')
])
```

**Note:** Dropout is only active during **training**. During inference (evaluation/prediction), Keras automatically scales the weights to account for the neurons that were dropped, ensuring your output remains consistent.

The model remembers what we discussed — it references overfitting when answering the dropout question.
Inspect the raw history object:


In [ ]:
print_history(tutor)


**👤 User:** What is overfitting in machine learning?

────────────────────────────────────────────────────────────


**🤖 Model:** **Overfitting** occurs when a model learns the training data "too well," capturing noise and random fluctuations rather than the underlying pattern.

As a result, the model performs exceptionally well on training data but fails to generalize to new, unseen data. It is essentially "memorizing" the data instead of learning the general rules.

**Key indicators:**
*   Low training error.
*   High testing/validation error.

**Common causes:**
*   Model is too complex (too many parameters).
*   Training dataset is too small or noisy.

To fix it, you typically use techniques like **regularization**, **pruning**, or **increasing the amount of training data**.

────────────────────────────────────────────────────────────


**👤 User:** How does dropout help with that?

...

────────────────────────────────────────────────────────────


**👤 User:** Can you give me a concrete code example of dropout in Keras?

────────────────────────────────────────────────────────────


**🤖 Model:** In Keras, you add dropout by inserting a `Dropout` layer between your dense layers. The argument (e.g., `0.5`) represents the fraction of input units to drop.

```python
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10,)),
    layers.Dropout(0.5),           # Randomly drops 50% of neurons
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),           # Randomly drops 30% of neurons
    layers.Dense(1, activation='sigmoid')
])
```

**Note:** Dropout is only active during **training**. During inference (evaluation/prediction), Keras automatically scales the weights to account for the neurons that were dropped, ensuring your output remains consistent.

────────────────────────────────────────────────────────────


## 6. Function Calling (Tool Use)

The model can decide to call a Python function rather than generating text.
This is how LLMs connect to real systems: databases, APIs, calculators, IoT devices.

The model receives function *signatures* (name + docstring + type hints) and chooses when to invoke them.


### 6.1 Smart Home — Lightbot

In [ ]:
def enable_lights() -> None:
    """Turn on the lighting system."""
    print("  💡 LIGHTBOT: Lights enabled.")

def set_light_color(rgb_hex: str) -> None:
    """Set the light color. rgb_hex is a 6-digit hex code, e.g. FF0000. Lights must be enabled first."""
    print(f"  🎨 LIGHTBOT: Color set to #{rgb_hex}.")

def stop_lights() -> None:
    """Turn off all lights."""
    print("  🌑 LIGHTBOT: Lights off.")

lightbot = client.chats.create(
    model=MODEL_ID,
    config={
        "tools": [enable_lights, set_light_color, stop_lights],
        "system_instruction": (
            "You are a smart home lighting assistant. "
            "Only perform lighting-related actions. Do nothing else."
        ),
    },
)

for msg in [
    "Turn the lights on please.",
    "I'd like a deep blue atmosphere.",
    "All done, lights off.",
]:
    print(f"👤 {msg}")
    resp = call_with_retry(lightbot.send_message, msg)
    time.sleep(13)
    if resp.text:
        print(f"🤖 {resp.text}")
    print()


👤 Turn the lights on please.
  💡 LIGHTBOT: Lights enabled.
🤖 OK. I've turned the lights on for you.

👤 I'd like a deep blue atmosphere.
  🎨 LIGHTBOT: Color set to #00008B.
🤖 OK. I've set the lights to a deep blue color for you.

👤 All done, lights off.
  🌑 LIGHTBOT: Lights off.
🤖 OK. I've turned the lights off for you.



The model automatically executes the functions. Inspect what happened under the hood:

In [ ]:
print_history(lightbot)


**👤 User:** Turn the lights on please.

────────────────────────────────────────────────────────────
  📞 Function call → enable_lights({})
────────────────────────────────────────────────────────────
  📬 Function response ← enable_lights: {'result': None}
────────────────────────────────────────────────────────────


**🤖 Model:** OK. I've turned the lights on for you.

────────────────────────────────────────────────────────────


**👤 User:** I'd like a deep blue atmosphere.

...

────────────────────────────────────────────────────────────


**👤 User:** All done, lights off.

────────────────────────────────────────────────────────────
  📞 Function call → stop_lights({})
────────────────────────────────────────────────────────────
  📬 Function response ← stop_lights: {'result': None}
────────────────────────────────────────────────────────────


**🤖 Model:** OK. I've turned the lights off for you.

────────────────────────────────────────────────────────────


### 6.2 Calculator — Grounding Math in Real Functions

In [ ]:
def add(a: float, b: float) -> float:
    """Returns a + b."""
    return a + b

def subtract(a: float, b: float) -> float:
    """Returns a - b."""
    return a - b

def multiply(a: float, b: float) -> float:
    """Returns a * b."""
    return a * b

def divide(a: float, b: float) -> float:
    """Returns a / b. Returns an error string if b is 0."""
    if b == 0:
        return "Error: division by zero"
    return a / b

calc_chat = client.chats.create(
    model=MODEL_ID,
    config={"tools": [add, subtract, multiply, divide]},
)

questions = [
    "If a cricket team scores 187 runs in 20 overs, what is their run rate per over?",
    "A data centre has 2048 GPUs. Each rack holds 8 GPUs. How many racks is that?",
]

for q in questions:
    print(f"👤 {q}")
    resp = call_with_retry(calc_chat.send_message, q)
    time.sleep(13)
    print(f"🤖 {resp.text}\n")


👤 If a cricket team scores 187 runs in 20 overs, what is their run rate per over?
🤖 The run rate for the cricket team is 9.35 runs per over.

👤 A data centre has 2048 GPUs. Each rack holds 8 GPUs. How many racks is that?
🤖 That is 256 racks.



In [ ]:
# See the function dispatch in the conversation history
print_history(calc_chat)


**👤 User:** If a cricket team scores 187 runs in 20 overs, what is their run rate per over?

────────────────────────────────────────────────────────────
  📞 Function call → divide({'a': 187, 'b': 20})
────────────────────────────────────────────────────────────
  📬 Function response ← divide: {'result': 9.35}
────────────────────────────────────────────────────────────


**🤖 Model:** The run rate for the cricket team is 9.35 runs per over.

────────────────────────────────────────────────────────────


**👤 User:** A data centre has 2048 GPUs. Each rack holds 8 GPUs. How many racks is that?

────────────────────────────────────────────────────────────
  📞 Function call → divide({'a': 2048, 'b': 8})
────────────────────────────────────────────────────────────
  📬 Function response ← divide: {'result': 256.0}
────────────────────────────────────────────────────────────


**🤖 Model:** That is 256 racks.

────────────────────────────────────────────────────────────


## 7. Applied NLP with Prompt Engineering

### 7.1 Sentiment Analysis

Zero-shot works. Few-shot gives you control over *your* label taxonomy.


In [ ]:
product_reviews = [
    "The battery life is incredible, easily lasts two days.",
    "Screen cracked on day 3 with normal use. Total waste of money.",
    "Average phone. Nothing stands out but nothing is broken either.",
    "Camera quality is breathtaking — best I've used.",
    "Customer support was rude and unhelpful when I reported the issue.",
]

# Zero-shot
zero_shot = (
    "Rate the sentiment of each review: POSITIVE, NEGATIVE, or NEUTRAL.\n"
    "Return only: <number>. <LABEL>\n\n"
    + "\n".join(f"{i+1}. {r}" for i, r in enumerate(product_reviews))
)
resp = call_with_retry(client.models.generate_content, model=MODEL_ID, contents=zero_shot,
                       config=types.GenerateContentConfig(temperature=0))
print("Zero-shot labels:\n", resp.text)


Zero-shot labels:
 1. POSITIVE
2. NEGATIVE
3. NEUTRAL
4. POSITIVE
5. NEGATIVE


In [ ]:
# Few-shot with custom labels for an e-commerce context
few_shot = """
Classify each review using exactly one of these labels:
  LOVE / HATE / MEH / COMPLAINT / PRAISE

Examples:
Review: "Works perfectly, no issues at all." → MEH
Review: "DO NOT BUY. Broke in a week." → HATE
Review: "I absolutely adore this product!" → LOVE

Classify:
1. The battery life is incredible, easily lasts two days.
2. Screen cracked on day 3 with normal use. Total waste of money.
3. Average phone. Nothing stands out but nothing is broken either.
4. Camera quality is breathtaking — best I've used.
5. Customer support was rude and unhelpful when I reported the issue.

Return only: <number>. <LABEL>
"""
resp = call_with_retry(client.models.generate_content, model=MODEL_ID, contents=few_shot,
                       config=types.GenerateContentConfig(temperature=0))
print("Few-shot custom labels:\n", resp.text)


Few-shot custom labels:
 1. PRAISE
2. HATE
3. MEH
4. LOVE
5. COMPLAINT


### 7.2 Text Summarisation — Output Format Control

Same article, three different prompt styles → three very different outputs.
This shows how `system_instruction` controls output *format*, not just *style*.


In [ ]:
article = """
India's renewable energy sector crossed a major milestone in 2025, surpassing
200 GW of installed capacity — the third-largest renewable portfolio in the world.
Solar power alone accounts for 85 GW, driven by large utility-scale projects in
Rajasthan and Gujarat as well as rapid growth in rooftop solar adoption.
Wind energy adds another 48 GW, concentrated in Tamil Nadu and Gujarat.
The government's target is 500 GW of non-fossil fuel capacity by 2030, requiring
an investment of over $200 billion. Challenges remain: grid integration of
intermittent sources, land acquisition delays, and financing gaps for smaller developers.
"""

summary_styles = {
    "Bullet-point (quick scan)": (
        "Summarise the article as 4 concise bullet points. Start each with •"
    ),
    "Executive summary (1 paragraph)": (
        "Write a formal one-paragraph executive summary suitable for a board presentation."
    ),
    "ELI5 (explain like I'm 5)": (
        "Explain this article to a 10-year-old who has never heard of renewable energy. "
        "Use simple words and a fun analogy."
    ),
}

for style, instruction in summary_styles.items():
    resp = call_with_retry(
        client.models.generate_content,
        model=MODEL_ID,
        contents=f"Article:\n{article}",
        config=types.GenerateContentConfig(system_instruction=instruction, temperature=0.2),
    )
    display(Markdown(f"**{style}**\n{resp.text}"))
    print()
    time.sleep(13)


**Bullet-point (quick scan)**
• India reached a milestone of 200 GW in renewable energy capacity, securing the third-largest portfolio globally.
• Solar power leads the sector with 85 GW, complemented by 48 GW of wind energy concentrated in key states.
• The government aims to achieve 500 GW of non-fossil fuel capacity by 2030, necessitating over $200 billion in investment.
• Growth faces significant hurdles, including grid integration issues, land acquisition delays, and financing difficulties for smaller projects.

**Executive summary (1 paragraph)**
In 2025, India solidified its position as a global leader in renewable energy by surpassing 200 GW of installed capacity, anchored by significant growth in solar and wind infrastructure. While this milestone marks substantial progress toward the national objective of 500 GW of non-fossil fuel capacity by 2030, achieving this target will necessitate an estimated $200 billion in capital investment. To sustain this momentum, the sector must strategically address critical operational hurdles, including grid integration complexities, land acquisition bottlenecks, and the financing constraints currently impacting smaller-scale developers.

**ELI5 (explain like I'm 5)**
Imagine you have a giant toy robot that needs batteries to move. Usually, to get those batteries, we have to dig deep into the ground to find "old-fashioned" fuel (like coal or oil). But here’s the problem: once we use that fuel, it’s gone forever, and it makes the air a bit dirty.

**Renewable energy** is like giving your robot a **magic backpack that never runs out of power.** Instead of digging in the ground, we use things that are always around us—like the bright sunshine and the blowing wind. 

Here is the cool news from India:

*   **The Big Milestone:** India just built enough of these "magic power machines" (solar panels and giant wind turbines) to power millions of homes. They are now the third-best in the whole world at doing this!
*   **Sun and Wind:** They are using the hot sun in places like Rajasthan and the strong winds in places like Gujarat to spin giant fans that make electricity. 
*   **The Goal:** They want to make even more of this clean energy by the year 2030. It’s like trying to upgrade your robot from a small battery to a super-powered engine that runs on sunshine and air!

**Is it easy? Not quite.**
Think of it like trying to build a massive LEGO castle. Sometimes it’s hard to find enough space for all the bricks (that’s the "land" problem), sometimes it’s expensive to buy all the pieces (the "money" problem), and sometimes the sun goes down or the wind stops blowing, so the robot has to figure out how to keep running when the "magic" isn't happening right that second.

But India is working hard to solve these puzzles so that, in the future, we can have all the power we need without ever running out or hurting the planet!

### 7.3 Named Entity Recognition (NER)

Prompt the model to extract structured entities. Parse the JSON and work with it directly.


In [ ]:
ner_texts = [
    "Sundar Pichai, CEO of Google, announced a new partnership with Samsung in Seoul last Monday.",
    "Reliance Industries signed a ₹12,000 crore deal with BP at the World Economic Forum in Davos.",
]

def build_ner_prompt(text):
    return (
        "Extract all named entities from the text and return ONLY valid JSON with these exact keys:"
        "persons, organisations, locations, monetary_values, dates_times"
        "Each key maps to a list of strings. No markdown, no explanation."
        "Text: " + text
    )

for text in ner_texts:
    print(f"Text: {text}")
    resp = call_with_retry(
        client.models.generate_content,
        model=MODEL_ID,
        contents=build_ner_prompt(text),
        config=types.GenerateContentConfig(temperature=0),
    )
    raw = resp.text.strip()
    if raw.startswith("```"):
        raw = raw.split("", 1)[1].rsplit("```", 1)[0].strip()
    entities = json.loads(raw)
    print(json.dumps(entities, indent=2, ensure_ascii=False))
    print("─" * 60)
    time.sleep(13)


Text: Sundar Pichai, CEO of Google, announced a new partnership with Samsung in Seoul last Monday.
{
  "persons": [
    "Sundar Pichai"
  ],
  "organisations": [
    "Google",
    "Samsung"
  ],
  "locations": [
    "Seoul"
  ],
  "monetary_values": [],
  "dates_times": [
    "last Monday"
  ]
}
────────────────────────────────────────────────────────────
Text: Reliance Industries signed a ₹12,000 crore deal with BP at the World Economic Forum in Davos.
{
  "persons": [],
  "organisations": [
    "Reliance Industries",
    "BP",
    "World Economic Forum"
  ],
  "locations": [
    "Davos"
  ],
  "monetary_values": [
    "₹12,000 crore"
  ],
  "dates_times": []
}
────────────────────────────────────────────────────────────


### 7.4 Machine Translation with Context

LLMs excel here because they understand *context* — the same word means different
things in different domains.


In [ ]:
def translate(text: str, target_lang: str, context: str = None, temperature: float = 0.1) -> str:
    instruction = 'You are a professional translator. Return only the translation, nothing else.'
    parts = ['Translate to ', target_lang]
    if context:
        parts.append(' using ' + context + ' terminology')
    parts.append(': ' + text)  # colon-space separator keeps prompt clean
    prompt = ''.join(parts)
    resp = call_with_retry(
        client.models.generate_content,
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=instruction, temperature=temperature),
    )
    return resp.text.strip()


In [ ]:
# Multi-language demo
sentence = "Artificial intelligence is transforming healthcare."
languages = ["Hindi", "French", "German", "Japanese", "Arabic"]

print(f"Original: {sentence}\n")
for lang in languages:
    result = call_with_retry(translate, sentence, lang)
    print(f"{lang:10}: {result}")
    time.sleep(13)


Original: Artificial intelligence is transforming healthcare.

Hindi     : कृत्रिम बुद्धिमत्ता स्वास्थ्य सेवा में बदलाव ला रही है।
French    : L'intelligence artificielle transforme les soins de santé.
German    : Künstliche Intelligenz verändert das Gesundheitswesen.
Japanese  : 人工知能が医療を変革している。
Arabic    : يُحدث الذكاء الاصطناعي تحولاً في الرعاية الصحية.


In [ ]:
# Word-sense disambiguation via context
ambiguous = [
    ("This bat is looking good.",   "zoology",      "EN -> FR"),
    ("This bat is looking good.",   "cricket",      "EN -> FR"),
    ("What is the cost of Apple?",  "stock market", "EN -> FR"),
    ("What is the cost of Apple?",  "grocery",      "EN -> FR"),
]

print("Word-sense disambiguation through context:\n")
for text, context, _ in ambiguous:
    result = call_with_retry(translate, text, "French", context=context)
    print(f"Text   : {text}")
    print(f"Context: {context}")
    print(f"French : {result}")
    print()
    time.sleep(13)


Word-sense disambiguation through context:

Text   : This bat is looking good.
Context: zoology
French : Ce chiroptère présente un bon état physiologique.

Text   : This bat is looking good.
Context: cricket
French : Cette batte a une belle prise en main.

Text   : What is the cost of Apple?
Context: stock market
French : Quel est le cours de l'action Apple ?

Text   : What is the cost of Apple?
Context: grocery
French : Quel est le prix des pommes ?



### 7.5 Domain-Specific Prompting Templates

A reusable pattern: build a prompt *template* per domain, fill in the text at call time.
The same idea applies to any NLP task — summarisation, classification, extraction.


In [ ]:
DOMAIN_TEMPLATES = {
    "medical": (
        "You are a certified medical translator. Ensure accurate clinical terminology, "
        "professional tone, and patient-safety considerations. Translate to {lang}:\n\n{text}"
    ),
    "legal": (
        "You are a legal translator. Preserve precise legal register and jurisdictional "
        "terminology. Translate to {lang}:\n\n{text}"
    ),
    "marketing": (
        "You are a marketing copywriter and translator. Keep the text persuasive and "
        "culturally resonant. Translate to {lang}:\n\n{text}"
    ),
    "casual": (
        "Translate this conversational text to {lang} using natural, informal language "
        "with appropriate local expressions:\n\n{text}"
    ),
}

def domain_translate(text: str, domain: str, target_lang: str = "Spanish") -> str:
    if domain not in DOMAIN_TEMPLATES:
        return f"Unknown domain. Choose from: {list(DOMAIN_TEMPLATES)}"
    prompt = DOMAIN_TEMPLATES[domain].format(lang=target_lang, text=text)
    resp = call_with_retry(
        client.models.generate_content,
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return resp.text.strip()

# Demo
test_text = "The patient presents with acute myocardial infarction requiring immediate intervention."

for domain in ["medical", "legal", "casual"]:
    print(f"Domain [{domain:8}]: {domain_translate(test_text, domain, 'Spanish')}")
    time.sleep(13)


Domain [medical ]: As a certified medical translator, I provide the following translation, ensuring clinical accuracy and professional register:

**"El paciente presenta un infarto agudo de miocardio que requiere intervención inmediata."**

### Clinical Notes:
*   **Terminology:** "Infarto agudo de miocardio" (IAM) is the standard clinical term used in Spanish-speaking medical settings for "acute myocardial infarction."
*   **Register:** The tone is formal and objective, suitable for medical records, clinical handovers, or professional communication.
*   **Patient Safety:** This translation maintains the urgency of the original English text, ensuring that the critical nature of the condition is clearly communicated to the clinical team.
Domain [legal   ]: In a legal or medico-legal context, the translation should maintain formal clinical terminology while ensuring the precision required for medical records or expert testimony.

**Recommended translation:**

> **"El paciente presenta un

## 8. Prompt Engineering — Best Practices

### Quick-reference: which technique to use?

| Situation | Technique |
|---|---|
| Clear, unambiguous task | Zero-shot |
| Custom labels / specific output format | Few-shot |
| Reasoning / multi-step problems | Chain-of-Thought |
| Downstream code needs the output | Structured output (JSON) |
| Consistent persona / constraints | System instruction |
| Conversational context matters | Multi-turn chat |
| Real-world actions / external data | Function calling |

---

### ✅ DO
1. **Be specific** — "Translate to Mexican Spanish (informal, street register)" beats "Translate to Spanish"
2. **Provide context** — domain, audience, purpose
3. **Set output format explicitly** — "Return a JSON object with keys: ...", "List only, no explanation"
4. **Use examples for custom taxonomies** — few-shot is free fine-tuning
5. **Set `temperature=0` for deterministic tasks** — classification, extraction, structured output
6. **Cap token output** — `max_output_tokens` prevents runaway costs and forces conciseness

### ❌ DON'T
1. **Be vague** — "Make it better", "Make it sound good"
2. **Overload one prompt** — split complex tasks into sequential calls
3. **Assume shared context** — the model doesn't know your business jargon unless you tell it
4. **Ignore cultural adaptation** — language ≠ culture
5. **Trust output blindly** — always validate structured output (try/except on `json.loads`)

---

### Reusable Prompt Template Pattern

```python
TEMPLATE = """
You are a {role}.
Task: {task}
Context: {context}
Output format: {format}

Input: {input}
"""

def run_template(role, task, context, format, input_text):
    prompt = TEMPLATE.format(
        role=role, task=task, context=context, format=format, input_text=input_text
    )
    resp = client.models.generate_content(model=MODEL_ID, contents=prompt)
    return resp.text
```

Parameterising prompts this way makes them testable, versionable, and easy to hand off to teammates.


## 9. Conclusion

### What we covered

| Section | Concept |
|---|---|
| 3 | API basics — `generate_content`, response object, system instruction |
| 4 | Zero-shot, role-based, temperature, few-shot, CoT, structured output |
| 5 | Multi-turn chat — stateful conversation with history |
| 6 | Function calling — LLMs invoking real Python functions |
| 7 | Applied NLP — sentiment, summarisation, NER, translation, domain templates |

### What to explore next

- **Gemini 2.5 Pro** — stronger reasoning, longer context window (1M tokens)
- **Multimodal inputs** — send images, PDFs, audio alongside text
- **Embeddings** — `client.models.embed_content()` for semantic search and RAG
- **Batch API** — async bulk jobs for cost savings
- **LangChain / LlamaIndex** — frameworks that wrap these primitives for larger applications

### Resources

- [Google AI Studio](https://aistudio.google.com/) — free API key, prompt playground
- [Gemini API docs](https://ai.google.dev/gemini-api/docs)
- [google-genai Python SDK](https://github.com/googleapis/python-genai)
- [Prompt Engineering Guide](https://www.promptingguide.ai/)
- [Google Gemini Cookbook](https://github.com/google-gemini/cookbook)

---
*Built for the BITS NLP Webinar — feel free to fork and experiment!*
